<a href="https://colab.research.google.com/github/gabrielbecmar1932/Ingenieria-Logistica/blob/main/TSP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ingeniería Logística, Grado en Ingeniería Informática, **ULL** , curso 2021-2022.
---
# The Traveling Salesman Problem (TSP)
---


El Problema del Viajante de Comercio ("Traveling Salesman Problem") busca un circuito que visite una vez cada uno de los nodos de un grafo, con el menos coste posible de recorrido del circuito.
El ejercicio es importante porque muestra que pueden existir varios modelos para un mismo problema, y que no todos los modelos de un problema requieren el mismo tiempo de cálculo para ser resueltos.
De este ejercicio sobre el TSP aprenderemos a reconocer modelos buenos y modelos malos.

In [ ]:
# Definimos el tamaño del problema
n   = 25
V   = range(n)        # Conjunto de localizaciones
V0  = range(1,n)     # Conjunto de localizaciones sin el depósito
EPS = 0.001

# Generamos puntos en el plano
import random
random.seed(1)
points   = [(random.randint(0,100),random.randint(0,100)) for i in V]



Definimos una función para dibujar los puntos y la ruta.

In [ ]:
import matplotlib.pyplot as plt

# Función que dibuja
def dibuja(selected):                #dibuja una ruta dada como una lista de arcos
    plt.plot([p[0] for p in points], [p[1] for p in points], 'ro')
    #plt.plot([p[0] for p in range(1)], [p[1] for p in range(1)], 'bo')
    for (i,j) in selected:
        plt.plot([points[i][0],points[j][0]], [points[i][1],points[j][1]], 'b-')
    plt.show()

dibuja({})


## Representación de ejemplos con distancias euclídeas

A fin de que se vea algo con sentido, vamos a colocar puntos en el plano y a utilizar las distancias euclídeas entre ellos. Esto genera matrices de costos simétricas, es decir, donde $c_{ij}=c_{ji}$. Centrándonos ahora exclusivamente en distancias Euclideas para con ello "ver" la solución.

Definimos una matriz de costos

In [ ]:
import math
import numpy as np
cost =  np.zeros((n,n))
# Matriz de costos
for i in V :
  for j in V:
    if j != i:
      dx = points[i][0] - points[j][0]
      dy = points[i][1] - points[j][1]
      # El costo es igual a la distancia euclídea redondeada
      cost[i,j] = math.floor(math.sqrt(dx*dx + dy*dy))
      # Con esta otra hacemos que la matriz de costos no sea simétrica
      # cost[i,j] = math.floor(math.sqrt(dx*dx + dy*dy)*random.uniform(1.0, 1.2))



## Cuatro modelos para el TSP

Usando OR-tools como marco de trabajo, GLOP como librería de programación lineal continua y CBC como librería de optimización lineal entera, mostramos cuatro modelos para el TSP:

In [ ]:
!pip install ortools
from ortools.linear_solver import pywraplp

In [ ]:
""" Definimos una fución que resuleve el TSP con cuatro modelos diferentes
 p = Con restricciones de precendencia
 u = Con variables de posción
 f = Con variables de flujo
 g = con variables multiflujo
 Además distinguimos si lo resuleve sin la condición de integrabilidad ('LP') o con la condición de integrabilidad ('IP')
 Llamamos a esta función ATSP porque resuelve el Asymetric TSP.
 En este estos ejemplos hemos generado los costos o distancias de forma simétrica pero en general c_ij no es igual a c_ji)"""

selected = {}

def ATSP(var,tipo):
    if tipo == 'LP':
        solver = pywraplp.Solver('ATSPxLP', pywraplp.Solver.GLOP_LINEAR_PROGRAMMING)
        x = { (i,j) : solver.NumVar(0.0, solver.infinity(), 'x[%i,%i]' % (i,j)) for i in V for j in V if i!=j }
    else:
        solver = pywraplp.Solver('ATSPxIP', pywraplp.Solver.CBC_MIXED_INTEGER_PROGRAMMING)
        x = { (i,j) : solver.BoolVar('x[%i,%i]' % (i,j)) for i in V for j in V if i!=j }

    solver.Minimize(solver.Sum( cost[i,j]*x[i,j] for i in V for j in V if i!=j ))
    [ solver.Add( solver.Sum( x[i,j] for j in V if j!=i) == 1 )  for i in V ]
    [ solver.Add( solver.Sum( x[i,j] for i in V if j!=i) == 1 )  for j in V ]

    if var == "p" :
        p = { (i,j) : solver.NumVar(0, solver.infinity(), 'p[%i,%i]' % (i,j)) for i in V0 for j in V0 if i!=j }
        [ solver.Add( p[i,j] + p[j,i] == 1 )          for i in V0 for j in V0 if j!=i ]
        [ solver.Add( p[i,j] + p[j,k] + p[k,i] <= 2 ) for i in V0 for j in V0 for k in V0 if i!=j and i!=k and j!=k ]
        [ solver.Add( p[i,j] >= x[i,j]  )             for i in V0 for j in V0 if j!=i ]

    if var == "u" :
        u = { i : solver.NumVar(0.0, solver.infinity(), 'u[%i]' % i) for i in V0 }
        [ solver.Add( u[j] >= u[i] + x[i,j] - (n-2)*(1-x[i,j]) + (n-3)*x[j,i]   ) for i in V0 for j in V0 if j!=i ]

    if var == "f" :
        f = { (i,j) : solver.NumVar(0.0, solver.infinity(), 'f[%i,%i]' % (i,j)) for i in V for j in V if i!=j }
        [ solver.Add( solver.Sum(f[j,i]-f[i,j] for j in V if j!=i) >= 1 )  for i in V0 ]
        [ solver.Add(            f[i,j] <= (n-1)*x[i,j]           )    for i in V for j in V if i!=j ]

    if var == "g" :
        f = { (i,j,k) : solver.NumVar(0.0, solver.infinity(), 'f[%i,%i,%i]' % (i,j,k)) for i in V for j in V for k in V0 if i!=j}
        [ solver.Add( solver.Sum(f[j,i,k]-f[i,j,k] for j in V if j!=i) == 0 )  for i in V0 for k in V0 if k!= i ]
        [ solver.Add( solver.Sum(f[j,i,i]-f[i,j,i] for j in V if j!=i) == 1 )  for i in V0 ]
        [ solver.Add(            f[i,j,k] <= x[i,j]           )   for i in V for j in V for k in V0 if i!=j ]

    solver.Solve()
    print('Modelo con ',var,' y ',tipo,' opt=', solver.Objective().Value(),' en ', solver.WallTime()/1000, " segundos")
    global selected
    selected = [(i,j) for i in V for j in V if i!=j if x[i,j].solution_value() > EPS]

    # Comentar si no se quiere que se saque todas esta información
    [print('x[',i,',',j,']=', x[i,j].solution_value()) for i in V for j in V  if i!=j if x[i,j].solution_value() > EPS ]


In [ ]:
ATSP('u','LP')

In [ ]:
ATSP('p','LP')
ATSP('u','LP')
ATSP('f','LP')
# Este ejemplo con las variables g puede tardar demasiado. Comentalo si no quieres esperar tanto
ATSP('g','LP')


ATSP('p','IP')
ATSP('u','IP')
ATSP('f','IP')
# Este ejemplo con las variables g puede tardar demasiado. Comentalo si no quieres esperar tanto
ATSP('g','IP')


Modelo con  p  y  LP  opt= 466.0  en  0.306  segundos
Modelo con  u  y  LP  opt= 466.0  en  0.043  segundos
Modelo con  f  y  LP  opt= 445.875  en  0.039  segundos
Modelo con  g  y  LP  opt= 490.0  en  13.659  segundos
Modelo con  p  y  IP  opt= 498.0  en  31.581  segundos
Modelo con  u  y  IP  opt= 498.0  en  7.589  segundos
Modelo con  f  y  IP  opt= 498.0  en  4.209  segundos
Modelo con  g  y  IP  opt= 498.0  en  111.13  segundos


Observa que cuando no se exige la condición de integrabilidad las soluciones de los modelos no son iguales. Cuando sí se exige la condición de integrabilidad el valor óptimo sí es el mismo (como cabía esperar).

También observa que los tiempos de ejecución son diferentes entre unos modelos y otros.
Se podría probar cambiando la semilla y/o tamaño para ver si este comportamiento es general para varios ejemplos diferentes.


In [ ]:
dibuja(selected)

## Modelo con generación dinámica de filas

Mostramos ahora un modelo no compacto, es decir, con un enorme número de restricciones que no pueden ser generadas todas juntas, pero que sí son muy simples de utilizar dinámicamente a medida que van siendo necesarias. Es la formulación propuesta por Dantzig, Fulkerson y Johnson (1954) para el TSP, y hace uso de las llamadas "restricciones de eliminación de subciclos" (SEC "Subtour Elimination Constraints"): $$\sum_{i,j\in S} x_{ij} \leq |S|-1 \qquad \forall S \subset V.$$

Estas desigualdades anteriores son equivalentes a $$\sum_{i\in S} \sum_{j\not\in S} x_{ij} \geq 1 \qquad \forall S \subset V.$$ Notemos que podríamos reducir la cantidad de desigualdades en ambas familias limitándonos sólo a los subconjuntos que no contienen un nodo cualquiera, por ejemplo, $S\subset V\setminus\{1\}$. No es una gran reducción, pero es válida.

También sería valido quedarnos con sólo las desigualdades en las que $|S| \leq n/2$.

In [ ]:
#Definimos la variables x como antes

solver = pywraplp.Solver('ATSPsec', pywraplp.Solver.CBC_MIXED_INTEGER_PROGRAMMING)
x = { (i,j) : solver.BoolVar('x[%i,%i]' % (i,j)) for i in V for j in V if i!=j }

solver.Minimize(solver.Sum( cost[i,j]*x[i,j] for i in V for j in V if i!=j ))


import networkx as nx
def SEC():
    G = nx.Graph()
    selected = [(i,j) for i in V for j in V if i!=j if x[i,j].solution_value() > EPS]
    G.add_edges_from( selected )
    Components = list(nx.connected_components(G))
    return(Components)

def ATSPsec():
    [ solver.Add( solver.Sum( x[i,j] for j in V if j!=i) == 1 )  for i in V ]
    [ solver.Add( solver.Sum( x[i,j] for i in V if j!=i) == 1 )  for j in V ]

    solver.Solve()
    time = solver.WallTime()/1000
    Comp = SEC()
    while len(Comp) > 1:
        # Comentar las 4 líneas siguientes si no se desea tanta información
        print('Valor objetivo= ', solver.Objective().Value())
        print(Comp)
        selected = [(i,j) for i in V for j in V if i!=j if x[i,j].solution_value() > EPS]
        dibuja(selected)
        for S in Comp:
            if(len(S) <= n/2) :
                solver.Add( solver.Sum( x[i,j] for i in S for j in S if j!=i) <= len(S)-1 )
        solver.Solve()
        time += solver.WallTime()/1000
        Comp = SEC()

    print('Valor óptimo= ', solver.Objective().Value(),' en ', time, " segundos")
    selected = [(i,j) for i in V for j in V if i!=j if x[i,j].solution_value() > EPS]
    dibuja(selected)

ATSPsec()


# TSP con beneficios (*TSP with profits*)

Este problema es una variante del TSP en la que no todas las ciudades (localizaciones) tienen que ser visitadas, pero al visitar una ciudad tenemos un beneficio por ser visitada.
### Características y parámetros
* $G=(V,A)$ grafo dirigido, donde $V=\{1,\ldots,n\}$ es el conjunto de ciudades y $A=\{(i,j): i, j \in V, i\neq j \}$ es el conjunto de arcos.
* Vamos a considerar que 1 es la ciudad inicial y siempre tiene que ser visitada.
* $p_i$ es el beneficio de visitar la ciudad $i$ que puede ser visitada como máximo una vez.
* $c_{ij} = c_a$ es el coste de ir de la ciudad $i$ a la ciudad $j$.
* El **Objetivo** es maximizar el beneficio total menos el costo de la ruta.

### Variables de decisión (para un modelo con variables flujo)

* $y_i = 1 $ si la ciudad $i$ es visitado y $y_i = 0 $ en otro caso, para todo $i \in V$.
* $x_{ij} = x_a = 1$ si el arco $a=(i,j)$ está en la ruta y vale 0 en otro caso, para todo $a=(i,j) \in A$.
* $f_a=f_{ij}$ es la cantidad de producto que va de $i$ a $j$, para todo $i,j \in V \setminus \{1\}$.

### Modelo con variables flujo para evitar subciclos

$$\max \sum_{i \in V} p_iy_i - \sum_{a\in A} c_a x_a $$

sujeto a:

$$x(\delta^+(i))=x(\delta^-(i))=y_i \quad\quad\quad \forall i\in V$$
$$ x_a\in\{0,1\} \quad\quad\quad \forall a\in A$$
$$y_i\in\{0,1\} \quad\quad\quad \forall i\in V$$
$$y_1=1 \quad\quad\quad .$$
$$ \sum_{a\in x(\delta^+(i))} f_a - \sum_{a\in x(\delta^-(i))}f_a= y_i \quad\quad\quad \forall i\in V\backslash\{1\}$$
$$ 0 \leq f_a \leq (n-1)x_a \quad\quad\quad \forall a\in A$$

## Programa

### Parámetros

In [ ]:
# Definimos el tamaño del problema
n   = 15
V   = range(n)        # Conjunto de localizaciones
V0  = range(1,n)     # Conjunto de localizaciones sin el depósito
EPS = 0.001

# Generamos puntos en el plano
import random
random.seed(1)
points   = [(random.randint(0,100),random.randint(0,100)) for i in V]

# Generamos los beneficios
profits = [random.randint(1,100) for i in V]
profits[0] = 0     # Consideramos que la ciudad de partida no da beneficios

# Costos
import math
import numpy as np
cost =  np.zeros((n,n))
# Matriz de costos
for i in V :
  for j in V:
    if j != i:
      dx = points[i][0] - points[j][0]
      dy = points[i][1] - points[j][1]
      # El costo es igual a la distancia euclídea redondeada
      cost[i,j] = math.floor(math.sqrt(dx*dx + dy*dy))
      # Con esta otra hacemos que la matriz de costos no sea simétrica
      # cost[i,j] = math.floor(math.sqrt(dx*dx + dy*dy)*random.uniform(1.0, 1.2))

# Función que dibuja
import matplotlib.pyplot as plt
#dibuja una ruta dada como una lista de arcos
def dibuja(selected):
  # Creamos el dibujo más grande
  plt.figure(figsize=(20,12))
  plt.xlabel("X coordinate", fontsize='16')
  plt.ylabel("Y coordinate", fontsize='16')
  plt.title('TSP with profits \n', fontsize='18')

  # Dibujamos los nodos con el depósito en azul
  plt.plot([p[0] for p in points], [p[1] for p in points], 'ro')
  plt.plot(points[0][0], points[0][1], 'bo')
  for i in V0 :
     plt.annotate('(%i,%i)'%(i, profits[i]), (points[i][0],points[i][1]+1),fontsize='10',color='red')

  for (i,j) in selected:
    plt.plot([points[i][0],points[j][0]], [points[i][1],points[j][1]], 'b-')
    plt.annotate('%i'%cost[i][j], (((points[i][0]+points[j][0])/2)+1,((points[i][1]+points[j][1])/2)+1),fontsize='10')

  plt.show()

dibuja({})

### Modelo

In [ ]:
# MODELO
!pip install ortools
from ortools.linear_solver import pywraplp

selected = {}

solver = pywraplp.Solver('TSPwithProfits', pywraplp.Solver.CBC_MIXED_INTEGER_PROGRAMMING)

# Variables de decisión
x = { (i,j) : solver.BoolVar('x[%i,%i]' % (i,j)) for i in V for j in V if i!=j }
y = { (i) : solver.BoolVar('y[%i]' % (i)) for i in V }
f = { (i,j) : solver.NumVar(0.0, solver.infinity(), 'f[%i,%i]' % (i,j)) for i in V for j in V if i!=j }

# Función objetivo
solver.Maximize( solver.Sum( profits[i] * y[i] for i in V) - solver.Sum( cost[i,j]*x[i,j] for i in V for j in V if i!=j ) )

# Restricciones
[ solver.Add( solver.Sum( x[i,j] for j in V if j!=i) == y[i] )  for i in V ]
[ solver.Add( solver.Sum( x[j,i] for j in V if j!=i) == y[i] )  for i in V ]
solver.Add(y[0] == 1)

[ solver.Add( solver.Sum(f[j,i]-f[i,j] for j in V if j!=i) == y[i] )  for i in V0 ]
[ solver.Add(            f[i,j] <= (n-1)*x[i,j]           )    for i in V for j in V if i!=j ]

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
  print('Valor óptimo= ', solver.Objective().Value(),' en ', solver.WallTime()/1000, " segundos")
  selected = [(i,j) for i in V for j in V if i!=j if x[i,j].solution_value() > EPS]
  dibuja(selected)
else:
    print('The problem does not have an optimal solution.')

